In [1]:
# Import necessary libraries
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from kedro.io import DataCatalog
import json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, TimestampType
import pandas_ta as ta
from pyspark.sql.functions import col, date_sub, current_date, to_timestamp

In [ ]:
%load_ext kedro.ipython

In [3]:
catalog.list()


[
    'access_token',
    'access_kinesis',
    'kinesis_client',
    'validated_access_token',
    'validated_websocket_data_NSE_INDEX_Nifty_50',
    'raw_data_NSE_INDEX_Nifty_50_1minute',
    'transformed_data_NSE_INDEX_Nifty_50_1minute@csv',
    'indicators_data_NSE_INDEX_Nifty_50_1minute@csv',
    'feature_engineering_data_NSE_INDEX_Nifty_50_1minute@csv',
    'feature_engineering_data_NSE_INDEX_Nifty_50_1minute@spark',
    'raw_data_NSE_INDEX_Nifty_50_30minute',
    'transformed_data_NSE_INDEX_Nifty_50_30minute@csv',
    'indicators_data_NSE_INDEX_Nifty_50_30minute@csv',
    'feature_engineering_data_NSE_INDEX_Nifty_50_30minute@csv',
    'feature_engineering_data_NSE_INDEX_Nifty_50_30minute@spark',
    'raw_data_NSE_INDEX_Nifty_50_day',
    'transformed_data_NSE_INDEX_Nifty_50_day@csv',
    'indicators_data_NSE_INDEX_Nifty_50_day@csv',
    'feature_engineering_data_NSE_INDEX_Nifty_50_day@csv',
    'feature_engineering_data_NSE_INDEX_Nifty_50_day@spark',
    'raw_data_NSE_INDEX_Nif

In [19]:
BNF_df = catalog.load("feature_engineering_data_NSE_INDEX_Nifty_Bank_1minute@spark")
print(type(BNF_df))

[07/28/24 16:14:25] INFO     Loading data from                                                  ]8;id=939348;file:///opt/anaconda3/envs/stock_AI/lib/python3.8/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=501418;file:///opt/anaconda3/envs/stock_AI/lib/python3.8/site-packages/kedro/io/data_catalog.py#508\508]8;;\
                             feature_engineering_data_NSE_INDEX_Nifty_Bank_1minute@spark                           
                             (SparkDataset)...                                                                     

<class 'pyspark.sql.dataframe.DataFrame'>


In [20]:
# List of relevant columns based on provided parameters
relevant_columns = [
    'Timestamp','High', 'Low', 'Close', 'Volume', 'Open',
    'weekday_num','month_num','is_weekly_expiration','is_monthly_expiration',
    'SMA', 'EMA', 'RSI', 
    'MACD', 'MACD_signal', 
    'BBANDS_upper', 'BBANDS_middle','BBANDS_lower',
    'High_next','Low_next'
]
# Subset the DataFrame
BNF_df_sub = BNF_df.select([col(c) for c in relevant_columns])

# List of columns to check for null values
na_columns_to_check = [
    'SMA', 'EMA', 'RSI', 
    'MACD', 'MACD_signal', 
    'BBANDS_upper', 'BBANDS_middle','BBANDS_lower',
    'High_next','Low_next'
]

# Remove rows with null values in the specified columns
train_BNF_df_cleaned = BNF_df_sub.dropna(subset=na_columns_to_check)

print("Subset process complete..")

Subset process complete..


In [21]:
from datetime import datetime, timedelta
# Convert 'Timestamp' to timestamp type if not already
train_BNF_df_cleaned = train_BNF_df_cleaned.withColumn("Timestamp", to_timestamp(col("Timestamp")))

# Define start and end dates for the dataset
min_date_row = train_BNF_df_cleaned.agg({"Timestamp": "min"}).collect()[0][0]
max_date_row = train_BNF_df_cleaned.agg({"Timestamp": "max"}).collect()[0][0]

# Print the available range of data for verification
print(f"Data available from {min_date_row} to {max_date_row}")

# Function to split data based on provided date ranges
def split_data(df, train_start_date, train_end_date, test_start_date, test_end_date):
    train_df = df.filter((col("Timestamp") >= train_start_date) & (col("Timestamp") <= train_end_date))
    test_df = df.filter((col("Timestamp") >= test_start_date) & (col("Timestamp") <= test_end_date))
    return train_df, test_df


# 1. Split for 4 months training and 2 weeks testing
train_end_4m = max_date_row - timedelta(weeks=2)
train_start_4m = train_end_4m - timedelta(weeks=16)  # 4 months for training

test_start_4m = train_end_4m
test_end_4m = max_date_row

train_df_4m, test_df_4m = split_data(train_BNF_df_cleaned, train_start_4m, train_end_4m, test_start_4m, test_end_4m)
print("4 Months Training, 2 Weeks Testing:")
print(f"Training Data: {train_df_4m.count()} records")
print(f"Testing Data: {test_df_4m.count()} records")

# 2. Split for 1 month training and 2 weeks testing
train_end_1m = max_date_row - timedelta(weeks=2)
train_start_1m = train_end_1m - timedelta(weeks=4)  # 1 month for training

test_start_1m = train_end_1m
test_end_1m = max_date_row

train_df_1m, test_df_1m = split_data(train_BNF_df_cleaned, train_start_1m, train_end_1m, test_start_1m, test_end_1m)
print("1 Month Training, 2 Weeks Testing:")
print(f"Training Data: {train_df_1m.count()} records")
print(f"Testing Data: {test_df_1m.count()} records")

# 3. Split for 5 months training and 1 month testing
train_end_5m = max_date_row - timedelta(days=30)
train_start_5m = train_end_5m - timedelta(weeks=21.7)  # 5 months for training

test_start_5m = train_end_5m
test_end_5m = max_date_row

train_df_5m, test_df_5m = split_data(train_BNF_df_cleaned, train_start_5m, train_end_5m, test_start_5m, test_end_5m)
print("5 Months Training, 1 Month Testing:")
print(f"Training Data: {train_df_5m.count()} records")
print(f"Testing Data: {test_df_5m.count()} records")

# 4. Split for 5 months training and 2 weeks testing
train_end_5m_2w = max_date_row - timedelta(weeks=2)
train_start_5m_2w = train_end_5m_2w - timedelta(weeks=21.7)  # 5 months for training

test_start_5m_2w = train_end_5m_2w
test_end_5m_2w = max_date_row

train_df_5m_2w, test_df_5m_2w = split_data(train_BNF_df_cleaned, train_start_5m_2w, train_end_5m_2w, test_start_5m_2w, test_end_5m_2w)
print("5 Months Training, 2 Weeks Testing:")
print(f"Training Data: {train_df_5m_2w.count()} records")
print(f"Testing Data: {test_df_5m_2w.count()} records")


Data available from 2024-01-31 09:16:00 to 2024-07-26 14:56:00
4 Months Training, 2 Weeks Testing:
Training Data: 27480 records
Testing Data: 3375 records
1 Month Training, 2 Weeks Testing:
Training Data: 7126 records
Testing Data: 3375 records
5 Months Training, 1 Month Testing:
Training Data: 36925 records
Testing Data: 7875 records
5 Months Training, 2 Weeks Testing:
Training Data: 38426 records
Testing Data: 3375 records


In [ ]:

# Define training and testing splits
def check_splits():
    # 1. Split for 4 months training and 2 weeks testing, and 1 month training and 2 weeks testing
    train_end_4m = date_sub(end_date, 14)  # Last 2 weeks as testing
    train_start_4m = date_sub(train_end_4m, 120)  # 4 months for training

    test_end_4m = date_sub(end_date, 0)
    test_start_4m = date_sub(train_end_4m, 0)

    train_df_4m, test_df_4m = split_data(train_BNF_df_cleaned, train_start_4m, train_end_4m, test_start_4m, test_end_4m)
    print("4 Months Training, 2 Weeks Testing:")
    print(f"Training Data: {train_df_4m.count()} records")
    print(f"Testing Data: {test_df_4m.count()} records")

    train_end_1m = date_sub(end_date, 14)  # Last 2 weeks as testing
    train_start_1m = date_sub(train_end_1m, 30)  # 1 month for training

    test_end_1m = date_sub(end_date, 0)
    test_start_1m = date_sub(train_end_1m, 0)

    train_df_1m, test_df_1m = split_data(train_BNF_df_cleaned, train_start_1m, train_end_1m, test_start_1m, test_end_1m)
    print("1 Month Training, 2 Weeks Testing:")
    print(f"Training Data: {train_df_1m.count()} records")
    print(f"Testing Data: {test_df_1m.count()} records")

    # 2. Split for 5 months training and 1 month testing
    train_end_5m = date_sub(end_date, 30)  # Last 1 month as testing
    train_start_5m = date_sub(train_end_5m, 150)  # 5 months for training

    test_end_5m = date_sub(end_date, 0)
    test_start_5m = date_sub(train_end_5m, 0)

    train_df_5m, test_df_5m = split_data(train_BNF_df_cleaned, train_start_5m, train_end_5m, test_start_5m, test_end_5m)
    print("5 Months Training, 1 Month Testing:")
    print(f"Training Data: {train_df_5m.count()} records")
    print(f"Testing Data: {test_df_5m.count()} records")

    # 3. Split for 5 months training and 2 weeks testing
    train_end_5m_2w = date_sub(end_date, 14)  # Last 2 weeks as testing
    train_start_5m_2w = date_sub(train_end_5m_2w, 150)  # 5 months for training

    test_end_5m_2w = date_sub(end_date, 0)
    test_start_5m_2w = date_sub(train_end_5m_2w, 0)

    train_df_5m_2w, test_df_5m_2w = split_data(train_BNF_df_cleaned, train_start_5m_2w, train_end_5m_2w, test_start_5m_2w, test_end_5m_2w)
    print("5 Months Training, 2 Weeks Testing:")
    print(f"Training Data: {train_df_5m_2w.count()} records")
    print(f"Testing Data: {test_df_5m_2w.count()} records")

check_splits()

In [26]:
# # Get a summary of the DataFrame
# summary = BNF_df_sub.describe()

# # Show the summary
# summary.show()

# Calculate the count of null values for each column
null_counts = BNF_df_cleaned.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in BNF_df_cleaned.columns])

# Show the null counts
null_counts.show()

+----+---+-----+------+----+---------+-----------+---------+--------------------+---------------------+---+---+---+----+-----------+------------+-------------+------------+---------+--------+
|High|Low|Close|Volume|Open|Timestamp|weekday_num|month_num|is_weekly_expiration|is_monthly_expiration|SMA|EMA|RSI|MACD|MACD_signal|BBANDS_upper|BBANDS_middle|BBANDS_lower|High_next|Low_next|
+----+---+-----+------+----+---------+-----------+---------+--------------------+---------------------+---+---+---+----+-----------+------------+-------------+------------+---------+--------+
|   0|  0|    0|     0|   0|        0|          0|        0|                   0|                    0|  0|  0|  0|   0|          0|           0|            0|           0|        0|       0|
+----+---+-----+------+----+---------+-----------+---------+--------------------+---------------------+---+---+---+----+-----------+------------+-------------+------------+---------+--------+



In [ ]:
# Initialize Spark Session
spark = SparkSession.builder.appName("StockPrediction").getOrCreate()

# Assuming BNF_df is already loaded
# BNF_df = ...

# List of relevant columns based on provided parameters
relevant_columns = [
    'AROON_Up', 'AROON_Dn', 'AROON_Sc',
    'BOP',
    'CCI',
    'DC_upper', 'DC_middle', 'DC_lower',
    'EMA',
    'RSI',
    'STOCH_k', 'STOCH_d',
    'ICHIMOKU_a', 'ICHIMOKU_b', 'ICHIMOKU_base', 'ICHIMOKU_span', 'ICHIMOKU_CS',
    'KAMA',
    'BBANDS_upper', 'BBANDS_middle', 'BBANDS_lower',
    'MACD', 'MACD_signal', 'MACD_diff',
    'CMO',
    'DPO',
    'HMA',
    'KAMA',
    'KC_upper', 'KC_middle', 'KC_lower',
    'MOM',
    'PPO',
    'ROC',
    'TRIMA',
    'UO',
    'WILLR',
    'ATR',
    'SMA',
    'WMA',
    'Z',
    'AO',
    'ERI_bull', 'ERI_bear',
    'FISHER', 'FISHER_signal',
    'High', 'Low', 'Close', 'Volume', 'Open'
]

# Subset the DataFrame
BNF_df = BNF_df.select([col(c) for c in relevant_columns])

# Remove rows with null values
BNF_df = BNF_df.na.drop()

# Display the DataFrame schema to confirm the relevant columns
BNF_df.printSchema()

# Split data into training and testing sets (80-20 split)
train_df, test_df = BNF_df.randomSplit([0.8, 0.2], seed=42)

# Display the count of training and testing sets
print("Training Data Count: ", train_df.count())
print("Testing Data Count: ", test_df.count())
